# Reducing US Traffic Accidents

## Overview
This notebook is an analysis of US Traffic Accidents from 2018-2023.

BLUF

Pre-lim Analysis questions
1. Where — which geographic areas have disproportionately high severity 3/4 accidents relative to total accidents? (infrastructure investment)
2. When — what time patterns correlate with severity 3/4 accidents? (targeted enforcement, campaigns)
3. What conditions — which weather or road feature combinations are most associated with severity 3/4 accidents? (safety treatments, standards)

## Business Understanding
Define the problem clearly and identify key questions that your analysis should answer. Determine how success will be measured and what insights would be valuable to stakeholders.

## Data Understanding
Exploring the US Accidents dataset to understand its structure, variables, scope, and limitations.

Identifying potential issues with the data and assess its suitability for addressing the business problem.

In [1]:
# load in required libraries
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

Since the dataset is too big for github, download the file from Kaggle, unzip and put file in data directory.
Dataset from [Kaggle](https://www.kaggle.com/datasets/sobhanmoosavi/us-accidents/data) and place into `data/`
Run cell below to convert .csv into .parquet file for faster loading

In [2]:
# read csv file and convert to .parquet for faster loading
# data_csv = pd.read_csv('data/US_Accidents_March23.csv')
# data_csv.write_parquet('data/US_Accidents_March23.parquet')

In [3]:
df = pd.read_parquet('data/US_Accidents_March23.parquet')
# copy data to keep raw and cleaned separate
clean_df = df.copy()

In [4]:
print(df.info())

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 7728394 entries, 0 to 7728393
Data columns (total 46 columns):
 #   Column                 Dtype  
---  ------                 -----  
 0   ID                     object 
 1   Source                 object 
 2   Severity               int64  
 3   Start_Time             object 
 4   End_Time               object 
 5   Start_Lat              float64
 6   Start_Lng              float64
 7   End_Lat                float64
 8   End_Lng                float64
 9   Distance(mi)           float64
 10  Description            object 
 11  Street                 object 
 12  City                   object 
 13  County                 object 
 14  State                  object 
 15  Zipcode                object 
 16  Country                object 
 17  Timezone               object 
 18  Airport_Code           object 
 19  Weather_Timestamp      object 
 20  Temperature(F)         float64
 21  Wind_Chill(F)          float64
 22  Humidity(%)       

### Columns Description
* ID: This is a unique identifier of the accident record.
* Source: Source of raw accident data
* Severity: Shows the severity of the accident, a number between 1 and 4, where 1 indicates the least impact on traffic (i.e., short delay as a result of the accident) and 4 indicates a significant impact on traffic (i.e., long delay).
* Start_Time: Shows start time of the accident in local time zone.
* End_Time: Shows end time of the accident in local time zone. End time here refers to when the impact of accident on traffic flow was dismissed.
* Start_Lat: Shows latitude in GPS coordinate of the start point.
* Start_Lng: Shows longitude in GPS coordinate of the start point.
* End_Lat: Shows latitude in GPS coordinate of the end point.
* End_Lng: Shows longitude in GPS coordinate of the end point.
* Distance(mi): The length of the road extent affected by the accident in miles.
* Description: Shows a human provided description of the accident.
* Street: Shows the street name in address field.
* City: Shows the city in address field.
* County: Shows the county in address field.
* State: Shows the state in address field.
* Zipcode: Shows the zipcode in address field.
* Country: Shows the country in address field.
* Timezone: Shows timezone based on the location of the accident (eastern, central, etc.).
* Airport_Code: Denotes an airport-based weather station which is the closest one to location of the accident.
* Weather_Timestamp: Shows the time-stamp of weather observation record (in local time).
* Temperature(F): Shows the temperature (in Fahrenheit).
* Wind_Chill(F): Shows the wind chill (in Fahrenheit).
* Humidity(%): Shows the humidity (in percentage).
* Pressure(in): Shows the air pressure (in inches).
* Visibility(mi): Shows visibility (in miles).
* Wind_Direction: Shows wind direction.
* Wind_Speed(mph): Shows wind speed (in miles per hour).
* Precipitation(in): Shows precipitation amount in inches, if there is any.
* Weather_Condition: Shows the weather condition (rain, snow, thunderstorm, fog, etc.)
* Amenity: A POI annotation which indicates presence of amenity in a nearby location.
* Bump: A POI annotation which indicates presence of speed bump or hump in a nearby location.
* Crossing: A POI annotation which indicates presence of crossing in a nearby location.
* Give_Way: A POI annotation which indicates presence of give_way in a nearby location.
* Junction: A POI annotation which indicates presence of junction in a nearby location.
* No_Exit: A POI annotation which indicates presence of no_exit in a nearby location.
* Railway: A POI annotation which indicates presence of railway in a nearby location.
* Roundabout: A POI annotation which indicates presence of roundabout in a nearby location.
* Station: A POI annotation which indicates presence of station in a nearby location.
* Stop: A POI annotation which indicates presence of stop in a nearby location.
* Traffic_Calming: A POI annotation which indicates presence of traffic_calming in a nearby location.
* Traffic_Signal: A POI annotation which indicates presence of traffic_signal in a nearby location.
* Turning_Loop: A POI annotation which indicates presence of turning_loop in a nearby location.
* Sunrise_Sunset: Shows the period of day (i.e. day or night) based on sunrise/sunset.
* Civil_Twilight: Shows the period of day (i.e. day or night) based on civil twilight.
* Nautical_Twilight: Shows the period of day (i.e. day or night) based on nautical twilight.
* Astronomical_Twilight: Shows the period of day (i.e. day or night) based on astronomical twilight.

In [5]:
# verbose setting for .head()
pd.set_option('display.max_columns', None)
pd.set_option('display.max_rows', None)

In [6]:
df.head()

,ID,Source,Severity,Start_Time,End_Time,Start_Lat,Start_Lng,End_Lat,End_Lng,Distance(mi),Description,Street,City,County,State,Zipcode,Country,Timezone,Airport_Code,Weather_Timestamp,Temperature(F),Wind_Chill(F),Humidity(%),Pressure(in),Visibility(mi),Wind_Direction,Wind_Speed(mph),Precipitation(in),Weather_Condition,Amenity,Bump,Crossing,Give_Way,Junction,No_Exit,Railway,Roundabout,Station,Stop,Traffic_Calming,Traffic_Signal,Turning_Loop,Sunrise_Sunset,Civil_Twilight,Nautical_Twilight,Astronomical_Twilight
0,A-1,Source2,3,2016-02-08 05:46:00,2016-02-08 11:00:00,39.865147,-84.058723,NaN,NaN,0.01,Right lane blocked due to accident on I-70 Eas...,I-70 E,Dayton,Montgomery,OH,45424,US,US/Eastern,KFFO,2016-02-08 05:58:00,36.9,NaN,91.0,29.68,10.0,Calm,NaN,0.02,Light Rain,False,False,False,False,False,False,False,False,False,False,False,False,False,Night,Night,Night,Night
1,A-2,Source2,2,2016-02-08 06:07:59,2016-02-08 06:37:59,39.928059,-82.831184,NaN,NaN,0.01,Accident on Brice Rd at Tussing Rd. Expect del...,Brice Rd,Reynoldsburg,Franklin,OH,43068-3402,US,US/Eastern,KCMH,2016-02-08 05:51:00,37.9,NaN,100.0,29.65,10.0,Calm,NaN,0.00,Light Rain,False,False,False,False,False,False,False,False,False,False,False,False,False,Night,Night,Night,Day
2,A-3,Source2,2,2016-02-08 06:49:27,2016-02-08 07:19:27,39.063148,-84.032608,NaN,NaN,0.01,Accident on OH-32 State Route 32 Westbound at ...,State Route 32,Williamsburg,Clermont,OH,45176,US,US/Eastern,KI69,2016-02-08 06:56:00,36.0,33.3,100.0,29.67,10.0,SW,3.5,NaN,Overcast,False,False,False,False,False,False,False,False,False,False,False,True,False,Night,Night,Day,Day
3,A-4,Source2,3,2016-02-08 07:23:34,2016-02-08 07:53:34,39.747753,-84.205582,NaN,NaN,0.01,Accident on I-75 Southbound at Exits 52 52B US...,I-75 S,Dayton,Montgomery,OH,45417,US,US/Eastern,KDAY,2016-02-08 07:38:00,35.1,31.0,96.0,29.64,9.0,SW,4.6,NaN,Mostly Cloudy,False,False,False,False,False,False,False,False,False,False,False,False,False,Night,Day,Day,Day
4,A-5,Source2,2,2016-02-08 07:39:07,2016-02-08 08:09:07,39.627781,-84.188354,NaN,NaN,0.01,Accident on McEwen Rd at OH-725 Miamisburg Cen...,Miamisburg Centerville Rd,Dayton,Montgomery,OH,45459,US,US/Eastern,KMGY,2016-02-08 07:53:00,36.0,33.3,89.0,29.65,6.0,SW,3.5,NaN,Mostly Cloudy,False,False,False,False,False,False,False,False,False,False,False,True,False,Day,Day,Day,Day


# Data Preparation
Clean and preprocess the data, handling missing values, outliers, and duplicates. Create derived features that might provide additional insights (e.g., time of day categories, weather condition groupings).

In [7]:
# show missing values as percent of dataset, sorted from highest to lowest
missing = (
    (clean_df.isna().sum() / len(clean_df))
        .mul(100)
        .sort_values(ascending=False)
        .loc[lambda pct_missing: pct_missing > 0]
        .rename("missing_%")
        .to_frame()
)
missing

,missing_%
End_Lat,44.029355
End_Lng,44.029355
Precipitation(in),28.512858
Wind_Chill(F),25.865904
Wind_Speed(mph),7.391355
Visibility(mi),2.291524
Wind_Direction,2.267043
Humidity(%),2.253301
Weather_Condition,2.244438
Temperature(F),2.120143


## Cleaning Strategy

**Geospatial**
* `End_Lat` / `End_Lng` (~50% missing): Dropped. Start coordinates combined with distance provide sufficient spatial context if needed.
* `Airport_Code`, `Street`, `Timezone` (small % missing): Rows dropped — imputation is impractical for these identifiers.
* `City`, `Zipcode`: Retained for high-accident-volume analysis.

**Weather Sensors**
* `Precipitation` (~28% missing): Rather than imputing — which would introduce artificial values where none likely existed — nulls are preserved and a binary flag `Precipitation_Recorded` is added to distinguish "no data" from "no precipitation."
* `Wind_Chill(F)` (~26% missing): A binary flag `Wind_Chill_Present` is added to indicate when wind chill is applicable (temp < 50°F and wind speed > 3 mph). Missing values are then imputed using the NOAA wind chill formula rather than a simple median, since wind chill is calculable from existing variables.
* `Wind_Speed` (~7% missing): Imputed with the **median**. Absence of wind is unlikely; sensor malfunction is the more plausible explanation.
* `Wind_Direction` (low % missing): Imputed with **state-grouped mode** (categorical).
* `Visibility(mi)`, `Humidity(%)`, `Pressure(in)`, `Temperature(F)` (low % missing): Imputed with **state-grouped median**. These are supporting variables with minimal missingness.

**Weather Condition**
* `Weather_Condition` (primary variable): Imputed with **state-grouped mode**. Being categorical, mode is appropriate; state grouping adds geographic specificity.

**Temporal / Lighting**
* `Sunrise_Sunset`, `Nautical_Twilight`, `Astronomical_Twilight`: Dropped — low analytical value.
* `Civil_Twilight`: Retained as a potentially insightful lighting indicator. The ~0.3% missing rows are dropped.

In [8]:
# dropping End_lat and End_Lng since it is missing around 50% of values
# drop `Airport_Code`, `Street`, `Timezone`
clean_df = clean_df.drop(columns = ['End_Lat', 'End_Lng','Airport_Code','Street','Timezone','Sunrise_Sunset', 'Nautical_Twilight', 'Astronomical_Twilight'])

In [9]:
# create flag for whether precipitation is recorded
clean_df['Precipitation_Recorded'] = clean_df['Precipitation(in)'].notna().astype(bool)

In [10]:
# create a boolean mask for rows where wind chill should be present (temp < 50F and wind speed > 3mph)
# but is missing. Use the mask to calculate and fill wind chill values using the NOAA formula, only for the relevant rows

# col for where wind chill should have been calculated
clean_df['Wind_Chill_Present'] = (clean_df['Wind_Speed(mph)'] > 3) & (clean_df['Temperature(F)'] < 50)

mask = clean_df['Wind_Chill(F)'].isna() & clean_df['Wind_Chill_Present']

clean_df.loc[mask, 'Wind_Chill(F)'] = (
    35.75 
    + 0.6215 * clean_df.loc[mask, 'Temperature(F)'] 
    - 35.75 * clean_df.loc[mask, 'Wind_Speed(mph)'] ** 0.16 
    + 0.4275 * clean_df.loc[mask, 'Temperature(F)'] ** 0.16
)

In [11]:
# handle missing values for wind speed, visiblity, humidity, pressure, temperature - impute median by state
# .transform here so that it matches the df shape
clean_df['Wind_Speed(mph)'] = clean_df.groupby('State')['Wind_Speed(mph)'].transform(lambda state_vals: state_vals.fillna(state_vals.median()))
clean_df['Visibility(mi)'] = clean_df.groupby('State')['Visibility(mi)'].transform(lambda state_vals: state_vals.fillna(state_vals.median()))
clean_df['Humidity(%)'] = clean_df.groupby('State')['Humidity(%)'].transform(lambda state_vals: state_vals.fillna(state_vals.median()))
clean_df['Pressure(in)'] = clean_df.groupby('State')['Pressure(in)'].transform(lambda state_vals: state_vals.fillna(state_vals.median()))
clean_df['Temperature(F)'] = clean_df.groupby('State')['Temperature(F)'].transform(lambda state_vals: state_vals.fillna(state_vals.median()))

# handle missing values for wind direction, weather condition impute mode for categorical
clean_df['Wind_Direction'] = clean_df.groupby('State')['Wind_Direction'].transform(lambda state_vals: state_vals.fillna(state_vals.mode()[0]))


In [ ]:
# missing row drops
clean_df=clean_df.dropna(subset=['Civil_Twilight','Zipcode','City'],axis=0)

### Value Standardization and Type Casting

Wind Direction
CALM > Calm
South > S
North > N
East > E
West > W
Variable > VAR

Weather Condition
Create categories to group similar weather conditions and also create a binary flag `is_windy` for any windy suffix
| Category | Examples |
|---|---|
| **Clear / Fair** | Fair, Clear |
| **Cloudy** | Mostly Cloudy, Partly Cloudy, Overcast, Scattered Clouds |
| **Fog / Mist** | Fog, Mist, Haze, Shallow Fog, Patches of Fog, Partial Fog |
| **Rain** | Light Rain, Rain, Heavy Rain, Drizzle, Light Drizzle, Heavy Drizzle, Freezing Rain |
| **Snow** | Light Snow, Snow, Heavy Snow, Snow Grains, Blowing Snow, Drifting Snow |
| **Winter Mix** | Wintry Mix, Snow and Sleet, Light Snow and Sleet, Sleet, Ice Pellets, Light Freezing Drizzle |
| **Thunderstorm** | Thunder, T-Storm, Thunderstorm, Heavy T-Storm, Light Rain with Thunder |
| **Severe / Hazardous** | Tornado, Funnel Cloud, Squalls, Hail, Small Hail, Duststorm |
| **Atmospheric** | Smoke, Sand, Blowing Dust, Widespread Dust, Volcanic Ash |


In [ ]:
print(clean_df['Wind_Direction'].value_counts())
print(clean_df['Weather_Condition'].value_counts())

In [ ]:
# standardize wind directions
wind_direction_map = {'Calm':'CALM',
         'East':'E',
         'North':'N',
         'South':'S',
         'West':'W',
         'Variable':'VAR',
        }
clean_df['Wind_Direction'] = clean_df['Wind_Direction'].replace(wind_direction_map)
clean_df['Wind_Direction'].value_counts()

In [ ]:
# binary flag for windy and strip from weather_condition
clean_df['Is_Windy'] = clean_df['Weather_Condition'].str.lower().str.contains('windy',regex=False, na=False)
clean_df['Weather_Condition'] = clean_df['Weather_Condition'].str.replace(r'\s*/\s*Windy', '', regex=True)


In [ ]:
# One-hot encode for different conditions
clean_df['Clear/Fair'] = clean_df['Weather_Condition'].str.contains('fair|clear', case = False).astype(bool)
clean_df['Cloudy'] = clean_df['Weather_Condition'].str.contains('cloudy|mostly cloudy|partly cloudy|overcast|scattered clouds', case = False, na=False).astype(bool)
clean_df['Fog/Mist'] = clean_df['Weather_Condition'].str.contains('fog|mist|haze|shallow fog|patches of fog|partial fog', case = False, na=False).astype(bool)
clean_df['Rain'] = clean_df['Weather_Condition'].str.contains('rain|light rain|heavy rain|drizzle|light drizzle|heavy drizzle|freezing rain', case = False, na=False).astype(bool)
clean_df['Snow'] = clean_df['Weather_Condition'].str.contains('snow|light snow|heavy snow|snow grains|blowing snow|drifting snow', case = False, na=False).astype(bool)
clean_df['Winter Mix'] = clean_df['Weather_Condition'].str.contains('wintry mix|snow and sleet|light snow and sleet|sleet|ice pellets|light freezing drizzle', case = False, na=False).astype(bool)
clean_df['Thunderstorm'] = clean_df['Weather_Condition'].str.contains('thunder|storm|t-storm|thunderstorm|heavy t-storm|light rain with thunder', case = False, na=False).astype(bool)
clean_df['Severe/Hazardous'] = clean_df['Weather_Condition'].str.contains('tornado|funnel cloud|squalls|hail|small hail|duststorm', case = False, na=False).astype(bool)
clean_df['Atmospheric Obscurations'] = clean_df['Weather_Condition'].str.contains('smoke|sand|blowing dust|widespread dust|volcanic ash', case = False, na=False).astype(bool)

In [ ]:
# If Weather_Condition == "N/A Precipitation" → Precipitation_Recorded = 0
# Then reclassify those rows' Weather_Condition into whatever the base condition actually is (likely Clear / Fair or Cloudy given the context)
# That way N/A Precipitation doesn't survive as its own weather category, and the precipitation signal is consistently handled by your flag.
mask = clean_df['Weather_Condition'] == 'N/A Precipitation'
clean_df.loc[mask & (clean_df['Precipitation(in)'] > 0), 'Precipitation_Recorded'] = True
clean_df.loc[mask & (clean_df['Precipitation(in)'].isna() | (clean_df['Precipitation(in)'] == 0)), 'Precipitation_Recorded'] = False

In [ ]:
# Start_Time, End_Time, Weather_Timestamp should be datetime for time-based analysis

clean_df['Start_Time'] =pd.to_datetime(clean_df['Start_Time'], format='mixed')
clean_df['End_Time'] =pd.to_datetime(clean_df['End_Time'], format='mixed')
clean_df['Weather_Timestamp']=pd.to_datetime(clean_df['Weather_Timestamp'], format='mixed')

# change to category dtype to save memory and speed up on groupby. good for 7M+ rows

cat_cols = ['State', 'Country', 'Wind_Direction', 
            'Weather_Condition', 'Civil_Twilight']
clean_df[cat_cols] = clean_df[cat_cols].astype('category')

In [ ]:
clean_df[clean_df['Weather_Condition'] == 'N/A Precipitation']['Precipitation(in)'].value_counts(dropna=False)

In [ ]:
# check clean_Df
print(clean_df.info())
print(clean_df.head())

In [ ]:
# hard bound outliers for humidity (>100 and <0), neg values for wind-speed, 
# temperature >134 and <-80, wind speed, outliers
# lowest wind-chill -103
# http://en.wikipedia.org/wiki/U.S._state_and_territory_temperature_extremes
# https://mountwashington.org/remembering-the-big-wind/
# https://en.wikipedia.org/wiki/Atmospheric_pressure#:~:text=The%20highest%20adjusted%2Dto%2Dsea,1%2C083.8%20hPa%20(32.005%20inHg).
# 
print('Num Humidity out of bound:',len(clean_df[ (clean_df['Humidity(%)'] > 100) | (clean_df['Humidity(%)'] <0)]))
print('Num Wind Speed out of bound:',len(clean_df[ (clean_df['Wind_Speed(mph)']<0) | (clean_df['Wind_Speed(mph)']>231)]))
print('Num Temperature out of bound:',len(clean_df[ (clean_df['Temperature(F)'] > 134) | (clean_df['Temperature(F)'] <-80)]))
print('Num Pressure out of bound:',len(clean_df[ (clean_df['Pressure(in)']<20) | (clean_df['Pressure(in)']>40)]))

In [ ]:
mask = ((clean_df['Wind_Speed(mph)']<0) | (clean_df['Wind_Speed(mph)']>231)) | \
((clean_df['Temperature(F)'] > 134) | (clean_df['Temperature(F)'] <-80))| \
((clean_df['Pressure(in)']<20) | (clean_df['Pressure(in)']>40) )

clean_df = clean_df[~mask]

In [ ]:
for attr in ['year', 'month', 'day', 'hour', 'weekday']:
    clean_df[f'Start_{attr}'] = getattr(clean_df['date_col'].dt, attr)

In [ ]:
# Total accidents by severity level (counts and percentages)
# Date range of the dataset
# Geographic coverage (how many states, cities)
# Summary stats for key numerical columns



Potential Areas of Analysis
* Spatial and Temporal Patterns: Consider examining when and where accidents most frequently occur. You might explore patterns by time of day, day of week, season, and geographic location. This type of analysis could potentially reveal critical hotspots and time periods requiring intervention.
* Environmental Factors: You could investigate how weather conditions correlate with accident rates. Consider analyzing how visibility, precipitation, temperature, and other environmental variables might affect driver behavior and road conditions.
* Infrastructure Considerations: One possible avenue is to identify specific road features associated with accident severity. This might include road design, signage, lighting, or other infrastructural elements that could contribute to or mitigate accident risk.
* Urban vs. Rural Comparison: You may want to compare accident patterns between urban and rural settings. These different environments likely present distinct challenges and risk factors that might require tailored safety approaches.

# Analysis/Statistical Analysis
Apply appropriate statistical methods to identify patterns, correlations, and significant factors. Test hypotheses about accident causes and contributing factors.

# Evaluation
Interpret the findings in the context of the business problem. Assess whether the analysis provides actionable insights and addresses the key questions identified in the business understanding phase.